<a href="https://colab.research.google.com/github/SaB-true/DataScienceToolsDeliverable/blob/main/Family_Office_Fund.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install yfinance

import time
import numpy as np
import yfinance as yf
import pandas as pd
from datetime import datetime
from datetime import date

In [ ]:
# A continuación, la ruta de acceso al dataset con los tickers a analizar
ruta = '/content/drive/MyDrive/RETO GESTION PORTAFOLIOS PYTHON EQUIPO 2/tickers_masterON.xlsx'

df = pd.read_excel(ruta)
df.head()

,Yahoo Ticker,Original Ticker,Country,Industry,Benchmark Ticker,Benchmark Name
0,AAPL,AAPL,USA,Technology (Large Cap),^NDX,NASDAQ-100 Index
1,MSFT,MSFT,USA,Technology (Large Cap),^NDX,NASDAQ-100 Index
2,NVDA,NVDA,USA,Technology (Large Cap),^NDX,NASDAQ-100 Index
3,AVGO,AVGO,USA,Technology (Large Cap),^NDX,NASDAQ-100 Index
4,ORCL,ORCL,USA,Technology (Large Cap),^GSPC,S&P 500 Index


In [ ]:
#Extraeremos los tickers de stocks y de benchmarks
tickers_stocks = df['Yahoo Ticker'].dropna().unique().tolist()
tickers_benchmarks = df['Benchmark Ticker'].dropna().unique().tolist()

# Los juntamos con set() para quedarnos solo con una lista grande que podamos
# usar para la busqueda masiva en yahoo finance
# Esto garantiza que cada benchmark se descargue una sola vez
tickers_totales = list(set(tickers_stocks + tickers_benchmarks))

print(f"Descargamos {len(tickers_totales)} tickers")

Descargamos 1360 tickers


In [ ]:
#Definimos el horizonte temporal (hasta la fecha actual 19 de abril de 2026)
start_date = "2021-01-01"
end_date = date.today().strftime('%Y-%m-%d')

# Descargamos los precios de cierre ('Close') con frecuencia mensual ('1mo')
# '1mo' descarga el precio del primer día de cada mes :D
data = yf.download(
    tickers_totales,
    start=start_date,
    end=end_date,
    interval="1mo"
)['Close']


/tmp/ipykernel_1067/1976513097.py:7: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(
[*********************100%***********************]  1360 of 1360 completed
ERROR:yfinance:
120 Failed downloads:
ERROR:yfinance:['SWN', 'MODN', 'ARNC', 'NWG.AX', 'WETF', 'HOUL', 'DISH', 'THS', 'DPW.DE', 'ENDP', 'BACHOCOB.MX', 'NRW.AX', 'IENOVA.MX', 'NWLI', 'NTCO3.SA', 'MWT.AX', 'GSANBOB1.MX', 'BRFS3.SA', 'SAFM', 'MAXCOMCP.MX', 'STOR', 'AZTECACP.MX', 'RYDER', 'PKI', 'HA', 'GPS', 'K', 'PARA', 'SRC', 'CPE', 'CPLE6.SA', 'COMP.L', 'KRA', 'VEDL', 'BDEV.L', 'CTB', 'RE', 'SNV', 'QCCPO.MX', 'CNSL', 'MIME', 'TPX', 'ELET3.SA', 'CCRO3.SA', 'PRFT', 'PAPPEL.MX', 'GFREGIO.MX', 'MEXCHEM.MX', 'MDC', 'VCI.PA', 'AMAM', 'FI', 'SQ2.AX', 'AMSWA', 'VAH.AX', 'Y', 'DRE', 'HES', 'FIBRAMQ.MX', 'DFS', 'ALL.PA', 'HSII', 'MFRISCOA.MX', 'MRO', 'MUEN.DE', 'LCI', 'IDP.AX', 'COV.DE', 'OMN', 'JBSS3.SA', 'FIBRAPL.MX', 'SCB.L', 'ORP.PA', 'ALFAA.MX', 'SFR', 'MMC', 'IDEALB1.MX', 'HSBC.L', 

In [ ]:
# Identificamos qué tickers realmente trajeron datos (limpieza de columnas vacías)
data_limpia = data.dropna(axis=1, how='all')

# Informamos cuáles tickers se perdieron
tickers_exitosos = data_limpia.columns.tolist()
tickers_fallidos = list(set(tickers_totales) - set(tickers_exitosos))

print(f"Descarga exitosa: {len(tickers_exitosos)} tickers.")
print(f"Tickers eliminados (deslistados/error): {len(tickers_fallidos)}")

Descarga exitosa: 1240 tickers.
Tickers eliminados (deslistados/error): 121


In [ ]:
#calculamos retornos logarítmicos
log_retornos = np.log(data_limpia).diff()

# borramos la primera fila que sabemos que no tiene datos de retornos:
log_retornos = log_retornos.iloc[1:]

print(f"Dimensiones: {log_retornos.shape}")
print(log_retornos.head())

/usr/local/lib/python3.12/dist-packages/pandas/core/internals/blocks.py:393: RuntimeWarning: invalid value encountered in log
  result = func(self.values, **kwargs)


Dimensiones: (65, 1240)
Ticker        1605.T    3086.T    3099.T    3289.T    3405.T    3407.T  \
Date                                                                     
2021-02-01  0.259181  0.163040  0.190852  0.119474  0.062304 -0.016014   
2021-03-01 -0.036368  0.050028  0.001286 -0.016654  0.057857  0.106276   
2021-04-01 -0.013316 -0.010521 -0.013107 -0.064076 -0.062061 -0.088486   
2021-05-01  0.009339  0.034030  0.029737  0.068444 -0.043916  0.045414   
2021-06-01  0.096155 -0.088364 -0.014112  0.027316 -0.062662  0.012781   

Ticker        3436.T    360.AX    3861.T    4004.T  ...     ^FCHI     ^FTSE  \
Date                                                ...                       
2021-02-01  0.096710  0.123794  0.059880  0.038111  ...  0.054778  0.011776   
2021-03-01  0.038746  0.091567  0.064911  0.194201  ...  0.061871  0.034890   
2021-04-01  0.114346  0.184056 -0.029049  0.049546  ...  0.032791  0.037451   
2021-05-01 -0.108426  0.005186 -0.059808  0.028297  ...  0.027

In [ ]:
# Media aritmética de los retornos logarítmicos (mensual)
retorno_prom_mensual = log_retornos.mean()

# Anualización de los retornos logarítmicos
retorno_prom_anual = retorno_prom_mensual * 12

print(retorno_prom_mensual.head())
print(retorno_prom_anual.head())

Ticker
1605.T    0.034899
3086.T    0.019961
3099.T    0.028271
3289.T    0.016761
3405.T    0.008894
dtype: float64
Ticker
1605.T    0.418792
3086.T    0.239529
3099.T    0.339258
3289.T    0.201126
3405.T    0.106723
dtype: float64


In [ ]:
#consultamos algún ticker para comprobar su retorno promedio mensual logaritmico
print(retorno_prom_mensual["^GSPC"])

0.01051788207260721


In [ ]:
# Creamos una lista vacía para guardar los resultados finales
resultados = []

# Empezamos a recorrer el archivo de excel 'df' (el que tenemos los tickers, países e industrias)
# 'index' es el número de fila y 'fila' contiene la información de esa fila específica
for index, fila in df.iterrows():

    # Extraemos los nombres de los tickers como texto (ejemplo: "AAPL" y "^GSPC")
    ticker_activo = fila['Yahoo Ticker']
    ticker_bench = fila['Benchmark Ticker']

    # VERIFICACIÓN DE SEGURIDAD:
    # Solo entramos si ambos tickers existen en nuestra base de datos descargada
    if ticker_activo in log_retornos.columns and ticker_bench in log_retornos.columns:

        # 'log_retornos' es tu tabla completa (1,240 columnas).
        # Al poner [ticker_activo], estamos "recortando" solo la columna de ese activo.
        # En Pandas, una columna suelta se llama "Series". Por eso le puse 'serie_activo'.
        serie_activo = log_retornos[ticker_activo] # Todos los retornos de, ej. Apple
        serie_bench = log_retornos[ticker_bench]   # Todos los retornos de, ej. el S&P500

        # 1. CÁLCULO DE RETORNOS (Lo que ganamos)
        # Sacamos el promedio de la columna (que ya son log returns) y lo multiplicamos por 12 para anualizarlo
        retorno_prom_mensual = serie_activo.mean()
        retorno_anual_geom = np.exp(retorno_prom_mensual * 12) - 1 # Pasamos de logarítmico a real

        # 2. CÁLCULO DE RIESGO (Qué tanto brinca el precio)
        # Varianza: Es el riesgo "puro" (la distancia de los datos al promedio)
        var_mensual = serie_activo.var()
        var_anual = var_mensual * 12 # La varianza se anualiza linealmente

        # Volatilidad (Desviación Estándar):
        desvest_mensual = serie_activo.std()
        desvest_anual = desvest_mensual * np.sqrt(12) # multiplicamos * raíz cuadrada de 12

        # 3. RELACIÓN CON EL MERCADO (Covarianza y Beta)
        cov_mensual = serie_activo.cov(serie_bench)
        cov_anual = cov_mensual * 12

        # Beta:
        beta = cov_mensual / serie_bench.var()

        # Guardamos todo este "paquete" de datos en nuestra lista
        resultados.append({
            'Ticker': ticker_activo,
            'Country': fila['Country'],
            'Benchmark': ticker_bench,
            'Ret_Mensual_log': retorno_prom_mensual,
            'Ret_Anual_geom': retorno_anual_geom,
            'Var_Anual': var_anual,
            'desvest_Anual': desvest_anual,
            'Cov_Anual': cov_anual,
            'Beta': beta
        })

# Al final, convertimos esa lista de "paquetes" en una nueva tabla limpia
df_analisis = pd.DataFrame(resultados)

# Mostramos los primeros resultados para ver que todo esté en orden
print(df_analisis.head())

/usr/local/lib/python3.12/dist-packages/pandas/core/nanops.py:1675: RuntimeWarning: Degrees of freedom <= 0 for slice
  return np.cov(a, b, ddof=ddof)[0, 1]
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2773: RuntimeWarning: divide by zero encountered in divide
  c *= np.true_divide(1, fact)
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2773: RuntimeWarning: invalid value encountered in multiply
  c *= np.true_divide(1, fact)


  Ticker Country Benchmark  Ret_Mensual_log  Ret_Anual_geom  Var_Anual  \
0   AAPL     USA      ^NDX         0.012404        0.160494   0.057852   
1   MSFT     USA      ^NDX         0.010633        0.136089   0.054771   
2   NVDA     USA      ^NDX         0.043437        0.684124   0.242006   
3   AVGO     USA      ^NDX         0.039676        0.609803   0.127761   
4   ORCL     USA     ^GSPC         0.018950        0.255326   0.145098   

   desvest_Anual  Cov_Anual      Beta  
0       0.240524   0.034550  0.882747  
1       0.234031   0.037779  0.965245  
2       0.491941   0.081061  2.071107  
3       0.357437   0.047621  1.216717  
4       0.380918   0.038031  1.586072  


In [ ]:

#Creamos una lista para guardar los datos que vamos a solicitar a yahoo finance
#metadatos = []

#print("Iniciando descarga de datos...")

#for t in tickers_exitosos: # Usamos los tickers que sí se descargaron completos
#    try:
#        asset = yf.Ticker(t)
#        info = asset.info
#
#        metadatos.append({
#            'Ticker': t,
#            'ShortName': info.get('shortName'),
#            'Sector': info.get('sector'),
#            'Industry': info.get('industry'),
#            'MarketCap': info.get('marketCap'), # Viene en la moneda original del ticker
#            'Currency': info.get('currency')
#        })
#    except:
#        continue
#
# Convertimos la lista en un data frame
#datos_tickers_meta = pd.DataFrame(metadatos)
#print(datos_tickers_meta)

Iniciando descarga de datos...
      Ticker                        ShortName              Sector  \
0     1605.T                INPEX CORPORATION              Energy   
1     3086.T         J FRONT RETAILING CO LTD   Consumer Cyclical   
2     3099.T   ISETAN MITSUKOSHI HOLDINGS LTD   Consumer Cyclical   
3     3289.T  TOKYU FUDOSAN HOLDINGS CORPORAT         Real Estate   
4     3405.T                       KURARAY CO     Basic Materials   
...      ...                              ...                 ...   
1115    TRIP                TripAdvisor, Inc.   Consumer Cyclical   
1116    TRMB                     Trimble Inc.          Technology   
1117     TRN         Trinity Industries, Inc.         Industrials   
1118    TRNO       Terreno Realty Corporation         Real Estate   
1119    TROW        T. Rowe Price Group, Inc.  Financial Services   

                                Industry     MarketCap Currency  
0                          Oil & Gas E&P  4.631276e+12      JPY  
1       

In [ ]:
df_merged = pd.merge(df_analisis, datos_tickers_meta, on='Ticker', how='inner')
print(df_merged)

         Ticker Country Benchmark  Ret_Mensual_log  Ret_Anual_geom  Var_Anual  \
0          AAPL     USA      ^NDX         0.012404        0.160494   0.057852   
1          MSFT     USA      ^NDX         0.010633        0.136089   0.054771   
2          NVDA     USA      ^NDX         0.043437        0.684124   0.242006   
3          AVGO     USA      ^NDX         0.039676        0.609803   0.127761   
4          ORCL     USA     ^GSPC         0.018950        0.255326   0.145098   
...         ...     ...       ...              ...             ...        ...   
1121   LREN3.SA  Brazil     ^BVSP        -0.006975       -0.080298   0.191414   
1122   MULT3.SA  Brazil     ^BVSP         0.012715        0.164835   0.076288   
1123  IGTI11.SA  Brazil     ^BVSP         0.011962        0.154360   0.054421   
1124   CYRE3.SA  Brazil     ^BVSP         0.008871        0.112321   0.162333   
1125   EZTC3.SA  Brazil     ^BVSP        -0.004214       -0.049309   0.188340   

      desvest_Anual  Cov_An

In [ ]:
Ahora, el Market cap debemos estandarizarlo y pasarlo a USD únicamente:

In [ ]:
# Definimos los pares necesarios basándonos en los países disponibles
# Nota: EUR y GBP suelen cotizarse como moneda/USD, los demás como USD/moneda
divisas_tickers = ["USDMXN=X", "USDJPY=X", "USDAUD=X", "USDBRL=X", "USDSEK=X", "USDINR=X", "EURUSD=X", "GBPUSD=X"]

# Descargamos el último precio de cierre
fx_data = yf.download(divisas_tickers, period="1d")['Close'].iloc[-1]

# Creamos un diccionario de conversión a USD (Factor por el cual multiplicar la moneda local)
# Si es USD/Moneda (como MXN), el factor es 1/precio. Si es Moneda/USD (como EUR), el factor es el precio.
cambio_a_usd = {
    'USD': 1.0,
    'MXN': 1 / fx_data['USDMXN=X'],
    'JPY': 1 / fx_data['USDJPY=X'],
    'AUD': 1 / fx_data['USDAUD=X'],
    'BRL': 1 / fx_data['USDBRL=X'],
    'SEK': 1 / fx_data['USDSEK=X'],
    'INR': 1 / fx_data['USDINR=X'],
    'EUR': fx_data['EURUSD=X'],
    'GBP': fx_data['GBPUSD=X']
}

/tmp/ipykernel_1067/2598511090.py:6: FutureWarning: YF.download() has changed argument auto_adjust default to True
  fx_data = yf.download(divisas_tickers, period="1d")['Close'].iloc[-1]
[*********************100%***********************]  8 of 8 completed


In [ ]:
# Definimos una función para aplicar el cambio según la moneda de cada ticker
def convertir_cap(fila):
    moneda = fila['Currency']
    mkt_cap_local = fila['MarketCap']

    # Obtenemos el factor del diccionario, si no existe usamos 1, pensando que es dolar probablemente
    factor = cambio_a_usd.get(moneda, 1.0)

    return mkt_cap_local * factor

# Creamos la nueva columna en el dataframe mergeado
df_merged['MarketCap_USD'] = df_merged.apply(convertir_cap, axis=1)

# Verificamos los resultados
print(df_merged)

         Ticker Country Benchmark  Ret_Mensual_log  Ret_Anual_geom  Var_Anual  \
0          AAPL     USA      ^NDX         0.012404        0.160494   0.057852   
1          MSFT     USA      ^NDX         0.010633        0.136089   0.054771   
2          NVDA     USA      ^NDX         0.043437        0.684124   0.242006   
3          AVGO     USA      ^NDX         0.039676        0.609803   0.127761   
4          ORCL     USA     ^GSPC         0.018950        0.255326   0.145098   
...         ...     ...       ...              ...             ...        ...   
1121   LREN3.SA  Brazil     ^BVSP        -0.006975       -0.080298   0.191414   
1122   MULT3.SA  Brazil     ^BVSP         0.012715        0.164835   0.076288   
1123  IGTI11.SA  Brazil     ^BVSP         0.011962        0.154360   0.054421   
1124   CYRE3.SA  Brazil     ^BVSP         0.008871        0.112321   0.162333   
1125   EZTC3.SA  Brazil     ^BVSP        -0.004214       -0.049309   0.188340   

      desvest_Anual  Cov_An

Ahora, investigamos la tasa risk free para cada país, la agregaremos a un diccionario y luego lo agregaremos con un merge al data frame de df_merged

In [ ]:
# Definimos las tasas que investigamos (Risk-Free Rates)
tasas_riskfree = {
    'USA': 0.0432,
    'Mexico': 0.0850,
    'Japan': 0.02398,
    'UK': 0.0476,
    'France': 0.0373,
    'Germany': 0.0296,
    'Brazil': 0.1359,
    'Australia': 0.0493,
    'Sweden': 0.0271,
    'India': 0.0691
}

# Creamos la columna en tu df_merged usando el mapeo por país
df_merged['Risk_Free_Rate'] = df_merged['Country'].map(tasas_riskfree)
print(df_merged)

         Ticker Country Benchmark  Ret_Mensual_log  Ret_Anual_geom  Var_Anual  \
0          AAPL     USA      ^NDX         0.012404        0.160494   0.057852   
1          MSFT     USA      ^NDX         0.010633        0.136089   0.054771   
2          NVDA     USA      ^NDX         0.043437        0.684124   0.242006   
3          AVGO     USA      ^NDX         0.039676        0.609803   0.127761   
4          ORCL     USA     ^GSPC         0.018950        0.255326   0.145098   
...         ...     ...       ...              ...             ...        ...   
1121   LREN3.SA  Brazil     ^BVSP        -0.006975       -0.080298   0.191414   
1122   MULT3.SA  Brazil     ^BVSP         0.012715        0.164835   0.076288   
1123  IGTI11.SA  Brazil     ^BVSP         0.011962        0.154360   0.054421   
1124   CYRE3.SA  Brazil     ^BVSP         0.008871        0.112321   0.162333   
1125   EZTC3.SA  Brazil     ^BVSP        -0.004214       -0.049309   0.188340   

      desvest_Anual  Cov_An

A continuación, buscaremos empezar a analizar los activos para finalmente quedarnos con 20 stocks como máximo. Asimismo, buscaremos calcular el ratio de sortino que nos indica qué acciones tienden a ser mas volátiles a la baja. Tambien revisaremos los retornos anuales promedio, la beta y correlaciones.

In [ ]:
# Función para calcular la volatilidad solo de los meses con caídas
def calcular_downside_std(Ticker):
    # Tomamos la columna de retornos del ticker en tu tabla original
    rets = log_retornos[Ticker]
    # Filtramos solo los retornos negativos
    neg_rets = rets[rets < 0]
    # Anualizamos la desviación estándar: std * sqrt(12)
    return neg_rets.std() * np.sqrt(12)

#Calculamos la volatilidad a la baja para cada activo
df_merged['Downside_Desvest'] = df_merged['Ticker'].apply(calcular_downside_std)

# 2. Calculamos el Sortino: (Retorno - Tasa Libre de Riesgo) / Volatilidad a la Baja
df_merged['Sortino_Ratio'] = (df_merged['Ret_Anual_geom'] - df_merged['Risk_Free_Rate']) / df_merged['Downside_Desvest']
print(df_merged)

         Ticker Country Benchmark  Ret_Mensual_log  Ret_Anual_geom  Var_Anual  \
0          AAPL     USA      ^NDX         0.012404        0.160494   0.057852   
1          MSFT     USA      ^NDX         0.010633        0.136089   0.054771   
2          NVDA     USA      ^NDX         0.043437        0.684124   0.242006   
3          AVGO     USA      ^NDX         0.039676        0.609803   0.127761   
4          ORCL     USA     ^GSPC         0.018950        0.255326   0.145098   
...         ...     ...       ...              ...             ...        ...   
1121   LREN3.SA  Brazil     ^BVSP        -0.006975       -0.080298   0.191414   
1122   MULT3.SA  Brazil     ^BVSP         0.012715        0.164835   0.076288   
1123  IGTI11.SA  Brazil     ^BVSP         0.011962        0.154360   0.054421   
1124   CYRE3.SA  Brazil     ^BVSP         0.008871        0.112321   0.162333   
1125   EZTC3.SA  Brazil     ^BVSP        -0.004214       -0.049309   0.188340   

      desvest_Anual  Cov_An

A continuación, presentamos el primer pool antes de generar una metodología de optimización, presentaremos 3 pools. El primero, buscando empresas con mkt cap mayor a 500 millones de dólares y los 20 mejores Sortino Ratios. El segundo, los 20 mejores sortino ratios y delimitando a 3 de cada sector como máximo. El tercero y último, lo mismo que el segundo pero excluyendo empresas de sectores poco amigables con ESG.

In [ ]:
# Filtro de Capitalización Institucional
df_pool1 = df_merged[df_merged['MarketCap_USD'] >= 500_000_000].copy()

# Seleccionamos los 20 mejores Sortinos cuidando la diversificación
# Aquí podrías agrupar por industria y tomar los top 2 para asegurar variedad
pool_1 = df_pool1.sort_values('Sortino_Ratio', ascending=False).head(20)

print("Tus 20 candidatos finales basados en Sortino y Calidad Fundamental:")
print(pool_1[['Ticker', 'Country', 'Industry', 'Sortino_Ratio', 'Ret_Anual_geom', 'Beta']])

Tus 20 candidatos finales basados en Sortino y Calidad Fundamental:
           Ticker    Country                         Industry  Sortino_Ratio  \
1051       DNL.AX  Australia              Specialty Chemicals      36.861771   
884        5803.T      Japan                    Conglomerates       7.145573   
735     SAAB-B.ST     Sweden              Aerospace & Defense       4.449452   
360           HWM        USA              Aerospace & Defense       4.204248   
885        5801.T      Japan     Electrical Equipment & Parts       3.824961   
874        7011.T      Japan   Specialty Industrial Machinery       3.686167   
861        8630.T      Japan  Insurance - Property & Casualty       3.607537   
806        8031.T      Japan                    Conglomerates       3.400053   
803        8058.T      Japan                    Conglomerates       3.266088   
627   CERAMICB.MX     Mexico                             None       3.142797   
1095      NTPC.NS      India   Utilities - Regulated

Pool 2: Seleccionamos los mejores Sortinos, pero limitamos a 3 por sector para que sea variado

In [ ]:
df_INSTITUTIONAL = df_merged[df_merged['MarketCap_USD'] >= 500_000_000].copy()
pool_2 = df_INSTITUTIONAL.sort_values('Sortino_Ratio', ascending=False).groupby('Sector').head(3).head(20)
print(pool_2[['Ticker', 'Country', 'Industry', 'Sortino_Ratio', 'Ret_Anual_geom', 'Beta']])


             Ticker    Country                             Industry  \
1051         DNL.AX  Australia                  Specialty Chemicals   
884          5803.T      Japan                        Conglomerates   
735       SAAB-B.ST     Sweden                  Aerospace & Defense   
360             HWM        USA                  Aerospace & Defense   
861          8630.T      Japan      Insurance - Property & Casualty   
1095        NTPC.NS      India       Utilities - Regulated Electric   
860          8725.T      Japan      Insurance - Property & Casualty   
3              AVGO        USA                       Semiconductors   
59               FN        USA                Electronic Components   
889          5802.T      Japan                           Auto Parts   
795          8306.T      Japan                  Banks - Diversified   
821          1605.T      Japan                        Oil & Gas E&P   
623       BAFARB.MX     Mexico                       Packaged Foods   
442   

Pool 3: Seleccionamos los mejores Sortinos, pero limitamos a 3 por sector para que sea variado. Sin embargo, removemos empresas con relación a energía y defensa. Se delimitó a máximo 10 empresas de USA y 3 de cada uno de los demás países.

In [ ]:
#lista de exclusión ESG
industrias_excluidas = [
    'Chemicals', 'Aerospace & Defense', 'Oil & Gas Exploration & Production',
    'Oil & Gas Integrated', 'Oil & Gas Midstream', 'Oil & Gas Refining & Marketing',
    'Tobacco', 'Specialty Chemicals'
]

# Filtro base (Market Cap y ESG)
df_base = df_merged[
    (df_merged['MarketCap_USD'] >= 500_000_000) &
    (~df_merged['Industry'].isin(industrias_excluidas))
].sort_values('Sortino_Ratio', ascending=False)

# Lógica de selección con límites de País y Sector
final_tickers = []
counts_country = {}
counts_sector = {}

for index, row in df_base.iterrows():
    if len(final_tickers) >= 20:
        break

    pais = row['Country']
    sector = row['Sector']

    # Definir límite de país
    limite_pais = 10 if pais == 'USA' else 3

    # Validar si el país y el sector tienen cupo
    if counts_country.get(pais, 0) < limite_pais and counts_sector.get(sector, 0) < 3:
        final_tickers.append(row)
        counts_country[pais] = counts_country.get(pais, 0) + 1
        counts_sector[sector] = counts_sector.get(sector, 0) + 1

# Convertimos los seleccionados en el DataFrame final
pool_3 = pd.DataFrame(final_tickers)

# 4. Visualización
print("Pool 3: Selección Balanceada por País y Sector (Filtro ESG)")
print(pool_3[['Ticker', 'Country', 'Sector', 'Sortino_Ratio', 'Ret_Anual_geom', 'Beta']])

# Verificación de conteos para tu reporte
print("\nDistribución por País:")
print(pool_3['Country'].value_counts())

Pool 3: Selección Balanceada por País y Sector (Filtro ESG)
             Ticker    Country                  Sector  Sortino_Ratio  \
884          5803.T      Japan             Industrials       7.145573   
885          5801.T      Japan             Industrials       3.824961   
874          7011.T      Japan             Industrials       3.686167   
627     CERAMICB.MX     Mexico                    None       3.142797   
1095        NTPC.NS      India               Utilities       3.057599   
3              AVGO        USA              Technology       2.958364   
59               FN        USA              Technology       2.934621   
623       BAFARB.MX     Mexico      Consumer Defensive       2.649329   
642     ACTINVRB.MX     Mexico      Financial Services       2.640528   
1081  BHARTIARTL.NS      India  Communication Services       2.638038   
1015         SIG.AX  Australia              Healthcare       2.599115   
17             KLAC        USA              Technology       2.4

A continuación, construiremos una función para buscar el target price de un stock:

In [ ]:
def obtener_views_analistas(lista_tickers):
    resultados = []

    print(f"Descargando proyecciones para {len(lista_tickers)} activos...")

    for t in lista_tickers:
        try:
            asset = yf.Ticker(t)
            info = asset.info

            # Obtenemos el Precio Objetivo (Promedio de analistas)
            target_price = info.get('targetMeanPrice')

            # Obtenemos el Precio Actual (Usamos 'currentPrice' o 'previousClose')
            actual_price = info.get('currentPrice') or info.get('previousClose')

            if target_price and actual_price:
                # Calculamos la View (Retorno esperado)
                # Fórmula: (Target / Actual) - 1
                view_retorno = (target_price / actual_price) - 1

                # Guardamos el número de analistas como medida de Confianza (Opcional)
                num_analist_opinions = info.get('numberOfAnalystOpinions', 0)

                resultados.append({
                    'Ticker': t,
                    'Precio_Actual': actual_price,
                    'Target_Price': target_price,
                    'View_Q': view_retorno,
                    'Num_Analistas': num_analist_opinions
                })
            else:
                print(f"Sin proyecciones para {t}")

            # Pequeña pausa para no saturar la API
            time.sleep(0.2)

        except Exception as e:
            print(f"Error en {t}: {e}")
            continue

    return pd.DataFrame(resultados)

# Ejemplo de uso con el Pool 3
#df_views_pool3 = obtener_views_analistas(pool_3['Ticker'].tolist())

Descargando proyecciones para 20 activos...
Sin proyecciones para CERAMICB.MX


In [ ]:
# Juntamos todos los tickers únicos de los 3 pools
todos_los_tickers_pools = list(set(pool_1['Ticker'].tolist() +
                             pool_2['Ticker'].tolist() +
                             pool_3['Ticker'].tolist()))

# Corremos la función una sola vez para todos
df_views_3pools = obtener_views_analistas(todos_los_tickers_pools)

# Creamos un diccionario para un mapeo rápido {Ticker: View_Q}
dict_views = df_views_3pools.set_index('Ticker')['View_Q'].to_dict()

Descargando proyecciones para 36 activos...
Sin proyecciones para CIEB.MX
Sin proyecciones para CERAMICB.MX


In [ ]:
# Aplicamos el mapeo a cada pool
pool_1['View_Q'] = pool_1['Ticker'].map(dict_views)
pool_2['View_Q'] = pool_2['Ticker'].map(dict_views)
pool_3['View_Q'] = pool_3['Ticker'].map(dict_views)


Pool 3 con Perspectiva de Analistas (Q):
           Ticker Country       Sector  Sortino_Ratio    View_Q
884        5803.T   Japan  Industrials       7.145573 -0.074537
885        5801.T   Japan  Industrials       3.824961 -0.139235
874        7011.T   Japan  Industrials       3.686167  0.227625
627   CERAMICB.MX  Mexico         None       3.142797       NaN
1095      NTPC.NS   India    Utilities       3.057599  0.078524
Activos sin proyección en Pool 3: 1


A continuación, debido a que en los pools tenemos acciones sin View o con View de retornos negativos, limpiaremos cada pool con la siguiente función:

In [ ]:
# Función de limpieza rápida
def limpiar_pool(df):
    # Eliminamos filas donde View_Q es NaN (sin cobertura de analistas)
    df_limpio = df.dropna(subset=['View_Q']).copy()

    # Eliminamos activos con perspectiva negativa (Target < Precio Actual)
    # Para tu fondo de 7-15 años, solo queremos activos con potencial de subida
    df_limpio = df_limpio[df_limpio['View_Q'] > 0]

    return df_limpio

# Aplicamos a nuestros 3 pools
pool_1 = limpiar_pool(pool_1)
pool_2 = limpiar_pool(pool_2)
pool_3 = limpiar_pool(pool_3)

print(f"Limpieza completada. Activos restantes en Pool 3: {len(pool_3)}")

Limpieza completada. Activos restantes en Pool 3: 12
